# 01 — Análise exploratória (EDA)

Roda localmente (kernel `.venv`) ou no Colab (depois do `00_colab_setup`).
Pré-requisitos: `python pathoshift.py prepare` e `python pathoshift.py features`.

Cada seção indica a aula de onde vem a técnica.

In [ ]:
import sys
from pathlib import Path

# Raiz do repositório (funciona rodando de notebooks/ ou da raiz, local ou no Colab)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import utils
from src.data import load_hf_dataset, read_image
from src.manifest import load_manifest, summarize

cfg = utils.load_config(ROOT / "configs" / "default.yaml")
utils.set_seed(cfg["seed"])
man = load_manifest(utils.resolve_path(cfg["paths"]["manifest"]))
ds = load_hf_dataset(cfg)

## 1. Contagens por centro e por fold

In [ ]:
display(summarize(man, "centre_id"))
display(summarize(man, "fold").rename(index={-1: "test_ext"}))

## 2. Exemplos por centro (mudança de domínio: cada hospital cora diferente)

In [ ]:
from src.viz import examples_by_centre
examples_by_centre(ds, man, n=6, seed=cfg["seed"]);

## 3. Histogramas HSV médios por centro — Aula 01 (histograma, HSV)

In [ ]:
from src.viz import mean_hsv_hist_by_centre
mean_hsv_hist_by_centre(ds, man, n=100, seed=cfg["seed"]);

## 4. Equalização de histograma no canal V — Aula 01

In [ ]:
from src.color import equalize_v
from src.viz import show_grid

ids = [int(man[man["centre_id"] == c].sample(1, random_state=cfg["seed"])["id"].iloc[0]) for c in range(6)]
imgs = [read_image(ds, i) for i in ids]
show_grid(imgs + [equalize_v(im) for im in imgs],
          [f"c{c} original" for c in range(6)] + [f"c{c} equalizada" for c in range(6)], ncols=6);

## 5. Sobel (bordas) — Aula 03

In [ ]:
from src.color import to_gray
from src.filters import gaussian_filter, sobel_magnitude

grays = [to_gray(im) for im in imgs]
show_grid(grays + [sobel_magnitude(gaussian_filter(g, 5)) for g in grays],
          ["cinza"] * 6 + ["|Sobel|"] * 6, ncols=6, cmap="gray");

## 6. Espectro de Fourier e passa-baixa Gaussiano — Aula 04
O círculo vermelho é o raio D0 usado na feature "energia de alta frequência" (tudo fora dele é alta frequência).

In [ ]:
from src.filters import gaussian_lowpass, high_freq_energy, log_spectrum

d0 = cfg["features"]["hf_radius"]
fig, axes = plt.subplots(3, 6, figsize=(13, 6.5))
for j, g in enumerate(grays):
    axes[0, j].imshow(g, cmap="gray"); axes[0, j].set_title(f"AF={high_freq_energy(g, d0):.3f}", fontsize=8)
    axes[1, j].imshow(log_spectrum(g), cmap="gray")
    axes[1, j].add_patch(plt.Circle((128, 128), d0, fill=False, color="red"))
    axes[2, j].imshow(gaussian_lowpass(g, d0), cmap="gray"); axes[2, j].set_title("passa-baixa", fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
fig.tight_layout()

## 7. Máscara de tecido — Aula 06 (limiarização, Otsu) e Aula 07 (abertura e fecho)

**Achado importante:** o Otsu calculado em cada patch, no canal S, não separa tecido de fundo.
A maioria dos patches quase não tem fundo, então as duas classes que o Otsu encontra viram
"núcleo x estroma". O histograma de S de muitos patches mostra o fundo concentrado em S < 20,
por isso usamos um **limiar global fixo** (`features.tissue_threshold` no config).

In [ ]:
from src.color import rgb_to_hsv

# Histograma de S somado em 300 patches do dev
hist = np.zeros(256)
for i in man[man["split"] == "dev"].sample(300, random_state=cfg["seed"])["id"]:
    hist += np.bincount(rgb_to_hsv(read_image(ds, i))[..., 1].ravel(), minlength=256)
plt.figure(figsize=(7, 3))
plt.bar(range(256), hist / hist.sum(), width=1)
plt.axvline(cfg["features"]["tissue_threshold"], color="red", label="limiar fixo")
plt.xlabel("S"); plt.ylabel("fração de pixels"); plt.legend(); plt.title("Histograma do canal S (300 patches)");

In [ ]:
from src.segmentation import tissue_mask

f = cfg["features"]
# escolhe 3 patches com bastante fundo (S baixo) + 3 patches comuns
amostra = man[man["split"] == "dev"].sample(400, random_state=1)
fundo = [(rgb_to_hsv(read_image(ds, i))[..., 1] < 20).mean() for i in amostra["id"]]
ids_mask = amostra["id"].to_numpy()[np.argsort(fundo)[-3:]].tolist() + ids[:3]
rows = []
for i in ids_mask:
    im = read_image(ds, i)
    rows += [im,
             tissue_mask(im, threshold=None, blur_sigma=f["tissue_blur_sigma"], morphology=False),
             tissue_mask(im, f["tissue_threshold"], f["tissue_blur_sigma"], morphology=False),
             tissue_mask(im, f["tissue_threshold"], f["tissue_blur_sigma"], f["tissue_kernel"])]
show_grid(rows, ["original", "Otsu no patch", "limiar fixo", "+ abertura e fecho"] * len(ids_mask),
          ncols=4, cmap="gray");

## 8. Máscara de núcleos — Aula 06 (Otsu só dentro do tecido, componentes conexos) e Aula 07 (abertura, borda)
Limitação: em tecido muito denso os núcleos se encostam e viram um único componente,
o que infla a "área média dos núcleos".

In [ ]:
from src.segmentation import extract_border, nuclei_mask

rows = []
for i in ids_mask:
    im = read_image(ds, i)
    tis = tissue_mask(im, f["tissue_threshold"], f["tissue_blur_sigma"], f["tissue_kernel"])
    antes, _ = nuclei_mask(im, tis, morphology=False)
    depois, areas = nuclei_mask(im, tis, f["nuclei_kernel"], f["nuclei_min_area"])
    contorno = im.copy()
    contorno[extract_border(depois) > 0] = (0, 255, 0)
    rows += [im, antes, depois, contorno]
show_grid(rows, ["original", "Otsu (antes)", "abertura + comp. conexos", "borda (Aula 07)"] * len(ids_mask),
          ncols=4, cmap="gray");

## 9. Distribuição das features por classe

In [ ]:
from src.features import load_features

feats = load_features(utils.resolve_path(cfg["paths"]["features"]), man)
dev = feats[feats["split"] == "dev"]
cols = ["sobel_densidade", "laplaciano_var", "energia_alta_freq",
        "nucleo_fracao_area", "nucleo_numero", "nucleo_area_media"]
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, c in zip(axes.ravel(), cols):
    ax.boxplot([dev.loc[dev["label"] == y, c] for y in (0, 1)], showfliers=False)
    ax.set_xticks([1, 2], ["normal", "tumor"])
    ax.set_title(c)
fig.tight_layout()

In [ ]:
# Mediana das mesmas features em cada centro (mudança de domínio também aparece nas features)
feats.groupby("centre_id")[cols].median().round(3)

## Conclusões
- TODO (grupo): diferenças de cor entre centros, features mais discriminativas, limitações das máscaras.